# Reformat SY Export

### Step 0: Set-Up
Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [1]:
from typing import Tuple

import numpy as np
import pandas as pd
import xarray as xr
from tqdm.auto import tqdm  # Progress bar

import climakitae as ck
from climakitae.explore.standard_year_profile import (
    get_climate_profile,
    export_profile_to_csv,
    set_profile_metadata,
    _format_based_on_structure,
    _construct_profile_dataframe,
    _create_simple_dataframe,
    _create_single_wl_multi_sim_dataframe,
    _create_multi_wl_single_sim_dataframe,
    _create_multi_wl_multi_sim_dataframe,
     _stack_profile_data
)
from climakitae.explore.typical_meteorological_year import TMY
from climakitae.core.data_interface import (
    get_data_options,
    get_subsetting_options,
    get_data,
)

import warnings
warnings.filterwarnings("ignore")


from unittest.mock import MagicMock, call, patch
import pytest

## Where is the issue?

### Scratch

In [2]:
time_delta = pd.date_range(
    "2020-01-01", periods=8760, freq="h"
)  # 1 year of hourly data
warming_levels = [1.5]
simulations = ["sim1"]

# Create test data with proper dimensions
data = np.random.rand(len(warming_levels), len(time_delta), len(simulations))

sample_data = xr.DataArray(
    data,
    dims=["warming_level", "time_delta", "simulation"],
    coords={
        "warming_level": warming_levels,
        "time_delta": time_delta,
        "simulation": simulations,
    },
    attrs={"units": "degC", "variable_id": "tasmax"},
)


In [3]:
sample_data

<xarray.DataArray (warming_level: 1, time_delta: 8760, simulation: 1)> Size: 70kB
array([[[0.28490905],
        [0.37435618],
        [0.32762491],
        ...,
        [0.124169  ],
        [0.37648993],
        [0.96987809]]])
Coordinates:
  * warming_level  (warming_level) float64 8B 1.5
  * time_delta     (time_delta) datetime64[ns] 70kB 2020-01-01 ... 2020-12-30...
  * simulation     (simulation) <U4 16B 'sim1'
Attributes:
    units:        degC
    variable_id:  tasmax

In [4]:
days_in_year = 365
hours_in_year = 8760

In [41]:
def og_compute_profile(data: xr.DataArray, days_in_year: int = 365, q=0.5) -> pd.DataFrame:
    """
    Calculates the standard year climate profile for warming level data using 8760
    analysis.

    This function handles global warming levels approach using time_delta coordinate.
    Processes all 30 years of warming level data centered around the year a warming level
    is reached, computes the specified quantile for each hour of the year across all years,
    then selects the actual data value closest to that quantile (not interpolated),
    and returns a characteristic profile of 8760 hours (one year) for each warming level
    and simulation combination.

    Parameters
    ----------
    data : xr.DataArray
        Hourly base-line subtracted data for one variable with warming_level,
        time_delta, and simulation dimensions. Expected to contain ~30 years
        (262,800 hours) of data for each warming level and simulation.

    days_in_year : int, optional
        Either 366 or 365, depending on whether or not the year is a leap year.
        Default to 365 days

    q : float, optional
        Quantile value for selecting representative values (0.0 to 1.0).
        Default is 0.5 (median).

    Returns
    -------
    pd.DataFrame
        Standard year table for each warming level and simulation,
        with days of year as the index and hour of day as the columns.
        Multi-index columns include Hour, Warming_Level, and Simulation dimensions.

    """
    # Check for simulation dimension
    has_simulation = "simulation" in data.dims
    if has_simulation:
        n_simulations = len(data.simulation)
        simulations = data.simulation.values
    else:
        n_simulations = 1
        simulations = [None]

    # Get all available time_delta data (all 30 years)
    hours_per_day = 24
    hours_per_year = 8760
    total_hours = len(data.time_delta)
    n_years = total_hours // hours_per_year

    print(f"      📊 Processing {total_hours:,} hours ({n_years} years) of data")
    print(f"      🎯 Computing {q*100:.0f}th percentile for each hour of year")

    # Create hour-of-year coordinate for all data (cycling through 1-8760)
    hour_of_year_all = np.tile(np.arange(1, hours_per_year + 1), n_years)[:total_hours]
    data = data.assign_coords(hour_of_year=("time_delta", hour_of_year_all))

    warming_levels = data.warming_level.values

    # Create helper function to extract meaningful simulation labels
    def _get_simulation_label(sim: str | int | None, sim_idx: int) -> str:
        """Extract meaningful simulation label from simulation identifier."""
        if sim is None:
            return f"Sim_{sim_idx+1}"

        sim_str = str(sim)
        if "WRF_" in sim_str:
            # Parse simulation name format: WRF_GCM_params_scenario
            # Example: WRF_CESM2_r11i1p1f1_historical+ssp245
            parts = sim_str.split("_")
            if len(parts) >= 4:
                gcm = parts[1]  # e.g., CESM2, CNRM-ESM2-1
                params = parts[2]  # e.g., r11i1p1f1
                scenario = parts[3]  # e.g., historical+ssp245

                # Extract SSP from scenario (e.g., ssp245 from historical+ssp245)
                if "ssp" in scenario:
                    ssp_part = scenario.split("ssp")[-1]  # Get part after 'ssp'
                    ssp = f"ssp{ssp_part}"
                else:
                    ssp = "hist"  # fallback for historical-only

                return f"{gcm}-{params}-{ssp}"
            elif len(parts) >= 2:
                # Fallback for shorter format
                return f"{parts[1]}-{sim_idx+1}"
            else:
                return f"Sim_{sim_idx+1}"
        else:
            # Ensure uniqueness by adding index for non-WRF format
            base_name = sim_str.split("_")[0] if "_" in sim_str else sim_str
            return f"{base_name}-{sim_idx+1}"

    # Process all data using quantile computation across years
    print(
        f"      ⚙️ Computing quantiles for {len(warming_levels)} warming level(s) and {n_simulations} simulation(s)"
    )

    # Eagerly compute all dask data at once (one round-trip to scheduler)
    if hasattr(data.data, "chunks"):
        print("      📥 Loading data into memory...")
        from dask.diagnostics import ProgressBar

        with ProgressBar():
            data = data.compute()

    # Initialize storage for profiles
    profile_data = {}

    # Progress tracking
    total_combinations = len(warming_levels) * n_simulations
    with tqdm(
        total=total_combinations,
        desc="      Computing profiles",
        unit="combo",
        leave=False,
    ) as pbar:

        for wl_idx, wl in enumerate(warming_levels):
            for sim_idx, sim in enumerate(simulations):
                # Get simulation label
                sim_label = _get_simulation_label(sim, sim_idx)

                # Select data for this warming level and simulation combination
                if has_simulation:
                    subset_data = data.isel(warming_level=wl_idx, simulation=sim_idx)
                else:
                    subset_data = data.isel(warming_level=wl_idx)

                # Vectorized quantile computation using numpy
                # Reshape raw values into (n_years, hours_per_year) then compute
                # the quantile across years for each hour-of-year position
                values = subset_data.values
                n_total = len(values)
                usable = (n_total // hours_per_year) * hours_per_year
                year_hour_matrix = values[:usable].reshape(-1, hours_per_year)

                # Compute quantile targets for each of the 8760 hour positions
                quantile_targets = np.nanquantile(
                    year_hour_matrix, q, axis=0
                )  # shape: (8760,)

                # For each hour position, find the actual year whose value is
                # closest to the quantile (avoids interpolation)
                diffs = np.abs(
                    year_hour_matrix - quantile_targets[np.newaxis, :]
                )  # (n_years, 8760)
                closest_year_idx = np.nanargmin(diffs, axis=0)  # (8760,)
                profile_1d = year_hour_matrix[
                    closest_year_idx, np.arange(hours_per_year)
                ]

                # Reshape to (days_in_year, 24) for the final DataFrame
                profile_reshaped = profile_1d[: days_in_year * hours_per_day].reshape(
                    days_in_year, hours_per_day
                )

                # Store the profile
                key = (f"WL_{wl}", sim_label)
                profile_data[key] = profile_reshaped

                pbar.update(1)

    # Create the multi-index DataFrame structure
    df_profile = _construct_profile_dataframe(
        profile_data=profile_data,
        warming_levels=warming_levels,
        simulations=simulations,
        sim_label_func=_get_simulation_label,
        days_in_year=days_in_year,
        hours_per_day=hours_per_day,
    )

    # Determine which formatting function to use based on the structure
    _format_based_on_structure(df_profile)

    # Prepare metadata dictionary
    metadata = {
        "quantile": q,
        "method": "8760 analysis - actual data closest to quantile across 30 years",
        "description": f"Climate profile computed using actual data values closest to the {q*100:.0f}th percentile of hourly data",
    }

    # Add original data attributes if available
    if hasattr(data, "attrs"):
        if "units" in data.attrs:
            metadata["units"] = data.attrs["units"]
        if "extended_description" in data.attrs:
            metadata["extended_description"] = data.attrs["extended_description"]
        if "variable_id" in data.attrs:
            metadata["variable_name"] = data.attrs["variable_id"]
        elif hasattr(data, "name") and data.name:
            metadata["variable_name"] = data.name

    # Set all metadata using the helper function
    set_profile_metadata(df_profile, metadata)

    print(f"      ✅ Profile computation complete! Final shape: {df_profile.shape}")
    print(
        f"         With index: {df_profile.index.name}, columns: {df_profile.columns.names}"
    )
    if hasattr(data, "attrs") and "units" in data.attrs:
        print(f"         Units: {data.attrs['units']}")

    return df_profile


In [6]:
og_result = og_compute_profile(sample_data, days_in_year=365, q=0.5)

      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 1 warming level(s) and 1 simulation(s)


      Computing profiles:   0%|          | 0/1 [00:00<?, ?combo/s]

      ✅ Profile computation complete! Final shape: (365, 24)
         With index: Day of Year, columns: ['Hour', 'Simulation']
         Units: degC


In [7]:
og_result

Hour,1,2,3,4,5,6,7,8,9,10,...,15,16,17,18,19,20,21,22,23,24
Simulation,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,...,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1
Day of Year,,,,,,,,,,,,,,,,,,,,,
Jan-01,0.284909,0.374356,0.327625,0.217581,0.447564,0.140308,0.228239,0.970584,0.455261,0.967948,...,0.576781,0.358951,0.139541,0.668101,0.157610,0.495002,0.846295,0.148226,0.572226,0.078581
Jan-02,0.215804,0.988162,0.961628,0.161085,0.062328,0.972985,0.665452,0.435237,0.875950,0.899584,...,0.596676,0.360329,0.497779,0.421680,0.417961,0.784459,0.297854,0.691384,0.326726,0.325282
Jan-03,0.878520,0.828144,0.791833,0.441220,0.531271,0.793358,0.342716,0.856939,0.321393,0.029700,...,0.359150,0.354190,0.363143,0.813298,0.071003,0.555304,0.694933,0.364252,0.764301,0.327185
Jan-04,0.873684,0.403706,0.500852,0.286907,0.539369,0.019945,0.803015,0.685255,0.945266,0.611118,...,0.477979,0.214016,0.393318,0.533189,0.193859,0.119434,0.232538,0.667300,0.323407,0.673279
Jan-05,0.793272,0.364364,0.992818,0.666313,0.494999,0.882504,0.371635,0.233016,0.009744,0.812105,...,0.918625,0.751004,0.522202,0.888429,0.286055,0.670929,0.498397,0.889332,0.640148,0.639184
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Dec-27,0.917440,0.354909,0.770736,0.938789,0.300360,0.611036,0.756181,0.507140,0.466965,0.890467,...,0.695169,0.728059,0.905861,0.285362,0.400174,0.227709,0.844026,0.468976,0.742184,0.831842
Dec-28,0.226959,0.916828,0.600240,0.288850,0.445494,0.048073,0.240279,0.651629,0.179247,0.080315,...,0.186092,0.121580,0.653678,0.804297,0.100637,0.574936,0.696902,0.589437,0.526592,0.195385


In [45]:
# Check for simulation dimension
has_simulation = "simulation" in sample_data.dims
if has_simulation:
    n_simulations = len(sample_data.simulation)
    simulations = sample_data.simulation.values
else:
    n_simulations = 1
    simulations = [None]

# Get all available time_delta data (all 30 years)
hours_per_day = 24
hours_per_year = 8760
total_hours = len(sample_data.time_delta)
n_years = total_hours // hours_per_year

warming_levels = sample_data.warming_level.values

# Create helper function to extract meaningful simulation labels
def _get_simulation_label(sim: str | int | None, sim_idx: int) -> str:
    """Extract meaningful simulation label from simulation identifier."""
    if sim is None:
        return f"Sim_{sim_idx+1}"

    sim_str = str(sim)
    if "WRF_" in sim_str:
        # Parse simulation name format: WRF_GCM_params_scenario
        # Example: WRF_CESM2_r11i1p1f1_historical+ssp245
        parts = sim_str.split("_")
        if len(parts) >= 4:
            gcm = parts[1]  # e.g., CESM2, CNRM-ESM2-1
            params = parts[2]  # e.g., r11i1p1f1
            scenario = parts[3]  # e.g., historical+ssp245

            # Extract SSP from scenario (e.g., ssp245 from historical+ssp245)
            if "ssp" in scenario:
                ssp_part = scenario.split("ssp")[-1]  # Get part after 'ssp'
                ssp = f"ssp{ssp_part}"
            else:
                ssp = "hist"  # fallback for historical-only

            return f"{gcm}-{params}-{ssp}"
        elif len(parts) >= 2:
            # Fallback for shorter format
            return f"{parts[1]}-{sim_idx+1}"
        else:
            return f"Sim_{sim_idx+1}"
    else:
        # Ensure uniqueness by adding index for non-WRF format
        base_name = sim_str.split("_")[0] if "_" in sim_str else sim_str
        return f"{base_name}-{sim_idx+1}"

NameError: name 'sample_data' is not defined

In [42]:
def compute_profile(data: xr.DataArray, days_in_year: int = 365, q=0.5) -> pd.DataFrame:
    """
    Calculates the standard year climate profile for warming level data using 8760
    analysis.

    This function handles global warming levels approach using time_delta coordinate.
    Processes all 30 years of warming level data centered around the year a warming level
    is reached, computes the specified quantile for each hour of the year across all years,
    then selects the actual data value closest to that quantile (not interpolated),
    and returns a characteristic profile of 8760 hours (one year) for each warming level
    and simulation combination.

    Parameters
    ----------
    data : xr.DataArray
        Hourly base-line subtracted data for one variable with warming_level,
        time_delta, and simulation dimensions. Expected to contain ~30 years
        (262,800 hours) of data for each warming level and simulation.

    days_in_year : int, optional
        Either 366 or 365, depending on whether or not the year is a leap year.
        Default to 365 days

    q : float, optional
        Quantile value for selecting representative values (0.0 to 1.0).
        Default is 0.5 (median).

    Returns
    -------
    pd.DataFrame
        Standard year table for each warming level and simulation,
        with days of year as the index and hour of day as the columns.
        Multi-index columns include Hour, Warming_Level, and Simulation dimensions.

    """
    # Check for simulation dimension
    has_simulation = "simulation" in data.dims
    if has_simulation:
        n_simulations = len(data.simulation)
        simulations = data.simulation.values
    else:
        n_simulations = 1
        simulations = [None]

    # Get all available time_delta data (all 30 years)
    hours_per_day = 24
    hours_per_year = 8760
    total_hours = len(data.time_delta)
    n_years = total_hours // hours_per_year

    print(f"      📊 Processing {total_hours:,} hours ({n_years} years) of data")
    print(f"      🎯 Computing {q*100:.0f}th percentile for each hour of year")

    # Create hour-of-year coordinate for all data (cycling through 1-8760)
    hour_of_year_all = np.tile(np.arange(1, hours_per_year + 1), n_years)[:total_hours]
    data = data.assign_coords(hour_of_year=("time_delta", hour_of_year_all))

    warming_levels = data.warming_level.values

    # Create helper function to extract meaningful simulation labels
    def _get_simulation_label(sim: str | int | None, sim_idx: int) -> str:
        """Extract meaningful simulation label from simulation identifier."""
        if sim is None:
            return f"Sim_{sim_idx+1}"

        sim_str = str(sim)
        if "WRF_" in sim_str:
            # Parse simulation name format: WRF_GCM_params_scenario
            # Example: WRF_CESM2_r11i1p1f1_historical+ssp245
            parts = sim_str.split("_")
            if len(parts) >= 4:
                gcm = parts[1]  # e.g., CESM2, CNRM-ESM2-1
                params = parts[2]  # e.g., r11i1p1f1
                scenario = parts[3]  # e.g., historical+ssp245

                # Extract SSP from scenario (e.g., ssp245 from historical+ssp245)
                if "ssp" in scenario:
                    ssp_part = scenario.split("ssp")[-1]  # Get part after 'ssp'
                    ssp = f"ssp{ssp_part}"
                else:
                    ssp = "hist"  # fallback for historical-only

                return f"{gcm}-{params}-{ssp}"
            elif len(parts) >= 2:
                # Fallback for shorter format
                return f"{parts[1]}-{sim_idx+1}"
            else:
                return f"Sim_{sim_idx+1}"
        else:
            # Ensure uniqueness by adding index for non-WRF format
            base_name = sim_str.split("_")[0] if "_" in sim_str else sim_str
            return f"{base_name}-{sim_idx+1}"

    # Process all data using quantile computation across years
    print(
        f"      ⚙️ Computing quantiles for {len(warming_levels)} warming level(s) and {n_simulations} simulation(s)"
    )

    # Eagerly compute all dask data at once (one round-trip to scheduler)
    if hasattr(data.data, "chunks"):
        print("      📥 Loading data into memory...")
        from dask.diagnostics import ProgressBar

        with ProgressBar():
            data = data.compute()

    # Initialize storage for profiles
    profile_data = {}
    #! -start
    profile_data_reshaped = {}
    profile_data_1d = {}
    #! -end

    # Progress tracking
    total_combinations = len(warming_levels) * n_simulations
    with tqdm(
        total=total_combinations,
        desc="      Computing profiles",
        unit="combo",
        leave=False,
    ) as pbar:

        for wl_idx, wl in enumerate(warming_levels):
            for sim_idx, sim in enumerate(simulations):
                # Get simulation label
                sim_label = _get_simulation_label(sim, sim_idx)

                # Select data for this warming level and simulation combination
                if has_simulation:
                    subset_data = data.isel(warming_level=wl_idx, simulation=sim_idx)
                else:
                    subset_data = data.isel(warming_level=wl_idx)

                # Vectorized quantile computation using numpy
                # Reshape raw values into (n_years, hours_per_year) then compute
                # the quantile across years for each hour-of-year position
                values = subset_data.values
                n_total = len(values)
                usable = (n_total // hours_per_year) * hours_per_year
                year_hour_matrix = values[:usable].reshape(-1, hours_per_year)

                # Compute quantile targets for each of the 8760 hour positions
                quantile_targets = np.nanquantile(
                    year_hour_matrix, q, axis=0
                )  # shape: (8760,)

                # For each hour position, find the actual year whose value is
                # closest to the quantile (avoids interpolation)
                diffs = np.abs(
                    year_hour_matrix - quantile_targets[np.newaxis, :]
                )  # (n_years, 8760)
                closest_year_idx = np.nanargmin(diffs, axis=0)  # (8760,)
                profile_1d = year_hour_matrix[
                    closest_year_idx, np.arange(hours_per_year)
                ]

                # Reshape to (days_in_year, 24) for the final DataFrame
                profile_reshaped = profile_1d[: days_in_year * hours_per_day].reshape(
                    days_in_year, hours_per_day
                )

                # Store the profile
                key = (f"WL_{wl}", sim_label)
                profile_data[key] = profile_reshaped
                #! -start
                profile_data_reshaped[key] = profile_reshaped
                profile_data_1d[key] = profile_1d
                #! -end

                pbar.update(1)

    # # Create the multi-index DataFrame structure
    # df_profile = _construct_profile_dataframe(
    #     profile_data=profile_data,
    #     warming_levels=warming_levels,
    #     simulations=simulations,
    #     sim_label_func=_get_simulation_label,
    #     days_in_year=days_in_year,
    #     hours_per_day=hours_per_day,
    # )
    # #! -start
    # df_profile_reshaped = _construct_profile_dataframe(
    #     profile_data=profile_data_reshaped,
    #     warming_levels=warming_levels,
    #     simulations=simulations,
    #     sim_label_func=_get_simulation_label,
    #     days_in_year=days_in_year,
    #     hours_per_day=hours_per_day,
    # )
    # df_profile_1d = _construct_profile_dataframe(
    #     profile_data=profile_data_1d,
    #     warming_levels=warming_levels,
    #     simulations=simulations,
    #     sim_label_func=_get_simulation_label,
    #     days_in_year=days_in_year,
    #     hours_per_day=hours_per_day,
    # )
    # #! -end

    # # Determine which formatting function to use based on the structure
    # _format_based_on_structure(df_profile)

    # #! -start
    # _format_based_on_structure(df_profile_reshaped)
    # _format_based_on_structure(df_profile_1d)
    # #! -end

    # # Prepare metadata dictionary
    # metadata = {
    #     "quantile": q,
    #     "method": "8760 analysis - actual data closest to quantile across 30 years",
    #     "description": f"Climate profile computed using actual data values closest to the {q*100:.0f}th percentile of hourly data",
    # }

    # # Add original data attributes if available
    # if hasattr(data, "attrs"):
    #     if "units" in data.attrs:
    #         metadata["units"] = data.attrs["units"]
    #     if "extended_description" in data.attrs:
    #         metadata["extended_description"] = data.attrs["extended_description"]
    #     if "variable_id" in data.attrs:
    #         metadata["variable_name"] = data.attrs["variable_id"]
    #     elif hasattr(data, "name") and data.name:
    #         metadata["variable_name"] = data.name

    # # Set all metadata using the helper function
    # set_profile_metadata(df_profile, metadata)

    # #! -start
    # set_profile_metadata(df_profile_reshaped, metadata)
    # set_profile_metadata(df_profile_1d, metadata)
    # #! -end

    # print(f"      ✅ Profile computation complete! Final shape: {df_profile.shape}")
    # print(
    #     f"         With index: {df_profile.index.name}, columns: {df_profile.columns.names}"
    # )
    # if hasattr(data, "attrs") and "units" in data.attrs:
    #     print(f"         Units: {data.attrs['units']}")

    return profile_data_reshaped, profile_data_1d

In [10]:
profile_data_reshaped, profile_data_1d = compute_profile(sample_data, days_in_year=365, q=0.5)

      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 1 warming level(s) and 1 simulation(s)


      Computing profiles:   0%|          | 0/1 [00:00<?, ?combo/s]

In [11]:
profile_data_reshaped

{('WL_1.5',
  'sim1-1'): array([[0.28490905, 0.37435618, 0.32762491, ..., 0.14822606, 0.57222603,
         0.07858108],
        [0.21580415, 0.98816207, 0.961628  , ..., 0.69138354, 0.32672576,
         0.32528247],
        [0.87851959, 0.82814379, 0.79183343, ..., 0.36425233, 0.76430065,
         0.32718517],
        ...,
        [0.28028121, 0.79481371, 0.42320208, ..., 0.1849836 , 0.03679062,
         0.56663945],
        [0.93417293, 0.24224297, 0.38644672, ..., 0.20563585, 0.6683758 ,
         0.29615814],
        [0.19827366, 0.86066776, 0.05606331, ..., 0.124169  , 0.37648993,
         0.96987809]])}

In [12]:
sub_reshaped = profile_data_reshaped[('WL_1.5',
  'sim1-1')]
print(len(sub_reshaped[1]))
print(len(profile_data_reshaped[('WL_1.5',
  'sim1-1')]))
sub_1d = profile_data_1d[('WL_1.5',
  'sim1-1')]
print(len(sub_1d))

24
365
8760


profile_data_1d is 8760 long, while profile_data_reshaped is a set of 365 arrays, each 24 long. 

In [13]:
def _construct_profile_dataframe(
    profile_data: dict,
    warming_levels: np.ndarray,
    simulations: list,
    sim_label_func: callable,
    days_in_year: int,
    hours_per_day: int,
) -> pd.DataFrame:
    """
    Construct a DataFrame from profile data based on warming level and simulation dimensions.

    Parameters
    ----------
    profile_data : dict
        Dictionary with (warming_level, simulation) keys and profile arrays as values
    warming_levels : np.ndarray
        Array of warming level values
    simulations : list
        List of simulation identifiers
    sim_label_func : callable
        Function to extract simulation labels
    days_in_year : int
        Number of days in the year (365 or 366)
    hours_per_day : int
        Number of hours per day (24)

    Returns
    -------
    pd.DataFrame
        Structured DataFrame with appropriate column structure based on dimensions
    """
    hours = np.arange(1, 25, 1)
    n_warming_levels = len(warming_levels)
    n_simulations = len(simulations)
    print(f"n_warming_levels: {n_warming_levels}")
    print(f"n_simulations: {n_simulations}")

    if n_warming_levels == 1 and n_simulations == 1:
        return _create_simple_dataframe(
            profile_data,
            warming_levels[0],
            simulations[0],
            sim_label_func,
            days_in_year,
            hours,
            hours_per_day,
        )
    # elif n_warming_levels == 1 and n_simulations > 1:
    #     return _create_single_wl_multi_sim_dataframe(
    #         profile_data,
    #         warming_levels[0],
    #         simulations,
    #         sim_label_func,
    #         days_in_year,
    #         hours,
    #         hours_per_day,
    #     )
    # elif n_warming_levels > 1 and n_simulations == 1:
    #     return _create_multi_wl_single_sim_dataframe(
    #         profile_data,
    #         warming_levels,
    #         simulations[0],
    #         sim_label_func,
    #         days_in_year,
    #         hours,
    #         hours_per_day,
    #     )
    # else:
    #     return _create_multi_wl_multi_sim_dataframe(
    #         profile_data,
    #         warming_levels,
    #         simulations,
    #         sim_label_func,
    #         days_in_year,
    #         hours,
    #         hours_per_day,
    #     )


In [14]:
# Create the multi-index DataFrame structure
df_profile_reshaped = _construct_profile_dataframe(
    profile_data=profile_data_reshaped,
    warming_levels=warming_levels,
    simulations=simulations,
    sim_label_func=_get_simulation_label,
    days_in_year=days_in_year,
    hours_per_day=hours_per_day,
)

n_warming_levels: 1
n_simulations: 1


In [15]:
profile_data=profile_data_reshaped
warming_levels=warming_levels
simulations=simulations
sim_label_func=_get_simulation_label
days_in_year=days_in_year
hours_per_day=hours_per_day
hours = np.arange(1, 25, 1)

### The root of the problem
_create_simple_dataframe, and its ilk, will need to be modified.

In [16]:
def _create_simple_dataframe(
    profile_data: dict,
    warming_level: float,
    simulation: any,
    sim_label_func: callable,
    days_in_year: int,
    hours: np.ndarray,
    hours_per_day: int,
) -> pd.DataFrame:
    """
    Create a simple DataFrame for single warming level and single simulation.

    Parameters
    ----------
    profile_data : dict
        Profile data dictionary
    warming_level : float
        Single warming level value
    simulation : any
        Single simulation identifier
    sim_label_func : callable
        Function to get simulation label
    days_in_year : int
        Number of days in year
    hours : np.ndarray
        Array of hour values
    hours_per_day : int
        Hours per day (24)

    Returns
    -------
    pd.DataFrame
        Simple DataFrame with hour columns
    """

    wl_key = warming_level
    sim_key = sim_label_func(simulation, 0)

    # Create MultiIndex columns
    col_tuples = [(hour, sim_key) for hour in hours]
    multi_cols = pd.MultiIndex.from_tuples(col_tuples, names=["Hour", "Simulation"])

    # Stack data
    all_data = _stack_profile_data(
        profile_data=profile_data,
        hours_per_day=hours_per_day,
        wl_names=[f"WL_{wl_key}"],
        sim_names=[sim_key],
        hour_first=True,
    )

    return pd.DataFrame(
        all_data,
        columns=multi_cols,
        index=np.arange(1, days_in_year + 1, 1),
    )


In [17]:
test_profile = _create_simple_dataframe(
            profile_data,
            warming_levels[0],
            simulations[0],
            sim_label_func,
            days_in_year,
            hours,
            hours_per_day,
        )

Exploding _create_simple_dataframe()

In [18]:
profile_data = profile_data_reshaped
#profile_data = profile_data_1d
warming_level = warming_levels[0]
simulation = simulations[0]
sim_label_func = sim_label_func
days_in_year = days_in_year
hours = hours
hours_per_day = hours_per_day 

In [19]:
wl_key = warming_level
sim_key = sim_label_func(simulation, 0)

# Create MultiIndex columns
col_tuples = [(hour, sim_key) for hour in hours]
multi_cols = pd.MultiIndex.from_tuples(col_tuples, names=["Hour", "Simulation"])

# Stack data
all_data = _stack_profile_data(
    profile_data=profile_data,
    hours_per_day=hours_per_day,
    wl_names=[f"WL_{wl_key}"],
    sim_names=[sim_key],
    hour_first=True,
)

pd.DataFrame(
    all_data,
    columns=multi_cols,
    index=np.arange(1, days_in_year + 1, 1),
)

Hour,1,2,3,4,5,6,7,8,9,10,...,15,16,17,18,19,20,21,22,23,24
Simulation,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,...,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1
1,0.284909,0.374356,0.327625,0.217581,0.447564,0.140308,0.228239,0.970584,0.455261,0.967948,...,0.576781,0.358951,0.139541,0.668101,0.157610,0.495002,0.846295,0.148226,0.572226,0.078581
2,0.215804,0.988162,0.961628,0.161085,0.062328,0.972985,0.665452,0.435237,0.875950,0.899584,...,0.596676,0.360329,0.497779,0.421680,0.417961,0.784459,0.297854,0.691384,0.326726,0.325282
3,0.878520,0.828144,0.791833,0.441220,0.531271,0.793358,0.342716,0.856939,0.321393,0.029700,...,0.359150,0.354190,0.363143,0.813298,0.071003,0.555304,0.694933,0.364252,0.764301,0.327185
4,0.873684,0.403706,0.500852,0.286907,0.539369,0.019945,0.803015,0.685255,0.945266,0.611118,...,0.477979,0.214016,0.393318,0.533189,0.193859,0.119434,0.232538,0.667300,0.323407,0.673279
5,0.793272,0.364364,0.992818,0.666313,0.494999,0.882504,0.371635,0.233016,0.009744,0.812105,...,0.918625,0.751004,0.522202,0.888429,0.286055,0.670929,0.498397,0.889332,0.640148,0.639184
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,0.917440,0.354909,0.770736,0.938789,0.300360,0.611036,0.756181,0.507140,0.466965,0.890467,...,0.695169,0.728059,0.905861,0.285362,0.400174,0.227709,0.844026,0.468976,0.742184,0.831842
362,0.226959,0.916828,0.600240,0.288850,0.445494,0.048073,0.240279,0.651629,0.179247,0.080315,...,0.186092,0.121580,0.653678,0.804297,0.100637,0.574936,0.696902,0.589437,0.526592,0.195385
363,0.280281,0.794814,0.423202,0.458799,0.266903,0.962506,0.553719,0.185994,0.653409,0.910185,...,0.610212,0.890713,0.636051,0.532630,0.676790,0.966322,0.034423,0.184984,0.036791,0.566639


Now to explode _stack_profile_data

In [20]:
profile_data = profile_data_reshaped
#profile_data = profile_data_1d
hours_per_day=hours_per_day
wl_names=[f"WL_{wl_key}"]
sim_names=[sim_key]
hour_first = True,
three_level = False

In [21]:
# all_data = []

# if three_level:
#     # For three-level index: iterate hour -> wl -> sim
#     for hour in range(hours_per_day):
#         for wl_name in wl_names:
#             for sim_name in sim_names:
#                 key = (wl_name, sim_name)
#                 all_data.append(profile_data[key][:, hour])
# elif hour_first:
#     # For two-level with hour first
#     for hour in range(hours_per_day):
#         for wl_name in wl_names:
#             for sim_name in sim_names:
#                 key = (wl_name, sim_name)
#                 all_data.append(profile_data[key][:, hour])
# else:
#     # For other two-level cases
#     for wl_name in wl_names:
#         for sim_name in sim_names:
#             for hour in range(hours_per_day):
#                 key = (wl_name, sim_name)
#                 all_data.append(profile_data[key][:, hour])

# np.column_stack(all_data)

In [22]:
all_data = []

for hour in range(hours_per_day):
    for wl_name in wl_names:
        for sim_name in sim_names:
            key = (wl_name, sim_name)
            all_data.append(profile_data[key][:, hour])

In [23]:
len(all_data[1])

365

In [24]:
stacked_data = np.column_stack(all_data)

In [25]:
final_df = pd.DataFrame(
    stacked_data,
    columns=multi_cols,
    index=np.arange(1, days_in_year + 1, 1),
)

In [26]:
final_df

Hour,1,2,3,4,5,6,7,8,9,10,...,15,16,17,18,19,20,21,22,23,24
Simulation,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,...,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1,sim1-1
1,0.284909,0.374356,0.327625,0.217581,0.447564,0.140308,0.228239,0.970584,0.455261,0.967948,...,0.576781,0.358951,0.139541,0.668101,0.157610,0.495002,0.846295,0.148226,0.572226,0.078581
2,0.215804,0.988162,0.961628,0.161085,0.062328,0.972985,0.665452,0.435237,0.875950,0.899584,...,0.596676,0.360329,0.497779,0.421680,0.417961,0.784459,0.297854,0.691384,0.326726,0.325282
3,0.878520,0.828144,0.791833,0.441220,0.531271,0.793358,0.342716,0.856939,0.321393,0.029700,...,0.359150,0.354190,0.363143,0.813298,0.071003,0.555304,0.694933,0.364252,0.764301,0.327185
4,0.873684,0.403706,0.500852,0.286907,0.539369,0.019945,0.803015,0.685255,0.945266,0.611118,...,0.477979,0.214016,0.393318,0.533189,0.193859,0.119434,0.232538,0.667300,0.323407,0.673279
5,0.793272,0.364364,0.992818,0.666313,0.494999,0.882504,0.371635,0.233016,0.009744,0.812105,...,0.918625,0.751004,0.522202,0.888429,0.286055,0.670929,0.498397,0.889332,0.640148,0.639184
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,0.917440,0.354909,0.770736,0.938789,0.300360,0.611036,0.756181,0.507140,0.466965,0.890467,...,0.695169,0.728059,0.905861,0.285362,0.400174,0.227709,0.844026,0.468976,0.742184,0.831842
362,0.226959,0.916828,0.600240,0.288850,0.445494,0.048073,0.240279,0.651629,0.179247,0.080315,...,0.186092,0.121580,0.653678,0.804297,0.100637,0.574936,0.696902,0.589437,0.526592,0.195385
363,0.280281,0.794814,0.423202,0.458799,0.266903,0.962506,0.553719,0.185994,0.653409,0.910185,...,0.610212,0.890713,0.636051,0.532630,0.676790,0.966322,0.034423,0.184984,0.036791,0.566639


Now to massage to handle profile_data_1d

In [27]:
wl_names[0]

'WL_1.5'

In [28]:
sim_names

['sim1-1']

In [29]:

final_df = pd.DataFrame(
    profile_data_1d[(wl_names[0],
  sim_names[0])],
    columns=[sim_key],
    index=np.arange(1, hours_in_year + 1, 1),
)

In [30]:
# we want the final result to look like this
final_df

,sim1-1
1,0.284909
2,0.374356
3,0.327625
4,0.217581
5,0.447564
...,...
8756,0.335014
8757,0.107789
8758,0.124169
8759,0.376490


#### Beyond simple dataframes

In [2]:
time_delta = pd.date_range(
    "2020-01-01", periods=8760, freq="h"
)  # 1 year of hourly data
warming_levels = [1.5, 2.0]
simulations = ["sim1"]

# Create test data with proper dimensions
data = np.random.rand(len(warming_levels), len(time_delta), len(simulations))

sample_data_multi_sim = xr.DataArray(
    data,
    dims=["warming_level", "time_delta", "simulation"],
    coords={
        "warming_level": warming_levels,
        "time_delta": time_delta,
        "simulation": simulations,
    },
    attrs={"units": "degC", "variable_id": "tasmax"},
)

#profile_data_reshaped_multi_sim, profile_data_1d_multi_sim = compute_profile(sample_data_multi_sim, days_in_year=365, q=0.5)

In [43]:
profile_data_reshaped_multi_wl, profile_data_1d_multi_wl = compute_profile(sample_data_multi_sim, days_in_year=365, q=0.5)

      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 2 warming level(s) and 1 simulation(s)


      Computing profiles:   0%|          | 0/2 [00:00<?, ?combo/s]

In [47]:
# Get all available time_delta data (all 30 years)
hours_per_day = 24
hours_per_year = 8760

# Create helper function to extract meaningful simulation labels
def _get_simulation_label(sim: str | int | None, sim_idx: int) -> str:
    """Extract meaningful simulation label from simulation identifier."""
    if sim is None:
        return f"Sim_{sim_idx+1}"

    sim_str = str(sim)
    if "WRF_" in sim_str:
        # Parse simulation name format: WRF_GCM_params_scenario
        # Example: WRF_CESM2_r11i1p1f1_historical+ssp245
        parts = sim_str.split("_")
        if len(parts) >= 4:
            gcm = parts[1]  # e.g., CESM2, CNRM-ESM2-1
            params = parts[2]  # e.g., r11i1p1f1
            scenario = parts[3]  # e.g., historical+ssp245

            # Extract SSP from scenario (e.g., ssp245 from historical+ssp245)
            if "ssp" in scenario:
                ssp_part = scenario.split("ssp")[-1]  # Get part after 'ssp'
                ssp = f"ssp{ssp_part}"
            else:
                ssp = "hist"  # fallback for historical-only

            return f"{gcm}-{params}-{ssp}"
        elif len(parts) >= 2:
            # Fallback for shorter format
            return f"{parts[1]}-{sim_idx+1}"
        else:
            return f"Sim_{sim_idx+1}"
    else:
        # Ensure uniqueness by adding index for non-WRF format
        base_name = sim_str.split("_")[0] if "_" in sim_str else sim_str
        return f"{base_name}-{sim_idx+1}"

In [48]:

profile_data = profile_data_reshaped_multi_wl
#profile_data = profile_data_1d_multi_sim

#! explode each element 

#!#!  start _construct_profile_dataframe
sim_label_func=_get_simulation_label
# df_profile = _construct_profile_dataframe(
#     profile_data=profile_data,
#     warming_levels=warming_levels,
#     simulations=simulations,
#     sim_label_func=_get_simulation_label,
#     days_in_year=days_in_year,
#     hours_per_day=hours_per_day,
# )

hours = np.arange(1, 25, 1)
n_warming_levels = len(warming_levels)
n_simulations = len(simulations)


#!#!#! start  _create_multi_wl_single_sim_dataframe
simulation = simulations[0]

# df_profile = _create_multi_wl_single_sim_dataframe(
        #     profile_data,
        #     warming_levels,
        #     simulations[0],
        #     sim_label_func,
        #     days_in_year,
        #     hours,
        #     hours_per_day,
        # )

sim_name = sim_label_func(simulation, 0)
wl_names = [f"WL_{wl}" for wl in warming_levels]

# Create MultiIndex columns
col_tuples = [(hour, wl_name) for hour in hours for wl_name in wl_names]
multi_cols = pd.MultiIndex.from_tuples(col_tuples, names=["Hour", "Warming_Level"])

# Stack data
sim_names = [sim_name]
# all_data = _stack_profile_data(
#     profile_data=profile_data,
#     hours_per_day=hours_per_day,
#     wl_names=wl_names,
#     sim_names=[sim_name],
#     hour_first=True,
# )
for hour in range(hours_per_day):
    for wl_name in wl_names:
        for sim_name in sim_names:
            key = (wl_name, sim_name)
            all_data.append(profile_data[key][:, hour])

df_profile = pd.DataFrame(
    all_data,
    columns=multi_cols,
    index=np.arange(1, days_in_year + 1, 1),
)

NameError: name 'all_data' is not defined

In [34]:
sim_names

['sim1-1', 'sim2-2']

In [35]:
# For two-level with hour first
for hour in range(hours_per_day):
    for wl_name in wl_names:
        for sim_name in sim_names:
            key = (wl_name, sim_name)
            all_data.append(profile_data[key][:, hour])


In [36]:
testing_all_data_og = all_data
profile_data_reshaped_multi_sim

{('WL_1.5',
  'sim1-1'): array([[0.40848229, 0.90451474, 0.25329887, ..., 0.76400352, 0.02069031,
         0.45816028],
        [0.76244866, 0.71711057, 0.98617209, ..., 0.02272513, 0.83899744,
         0.30384091],
        [0.71160393, 0.11323741, 0.93017148, ..., 0.43216219, 0.9729976 ,
         0.52674926],
        ...,
        [0.30892793, 0.31205013, 0.6334405 , ..., 0.9584521 , 0.92482945,
         0.39671696],
        [0.22665784, 0.10141079, 0.38188637, ..., 0.23732809, 0.91748156,
         0.98415018],
        [0.44245287, 0.85468285, 0.40008148, ..., 0.81890588, 0.4658208 ,
         0.26997238]]),
 ('WL_1.5',
  'sim2-2'): array([[0.17817066, 0.57930304, 0.02862475, ..., 0.14440423, 0.9217287 ,
         0.62427049],
        [0.76912623, 0.87079794, 0.69776514, ..., 0.31219029, 0.76921168,
         0.10195329],
        [0.28762032, 0.87607815, 0.60260187, ..., 0.82725534, 0.04341932,
         0.41990886],
        ...,
        [0.67033454, 0.74723799, 0.14868521, ..., 0.96992382

In [37]:
testing_all_data_og

[array([0.40848229, 0.76244866, 0.71160393, 0.67095389, 0.22245696,
        0.40007996, 0.72866621, 0.48887204, 0.0736698 , 0.06597339,
        0.51799581, 0.36124138, 0.42717892, 0.69769111, 0.74895631,
        0.57731601, 0.30352904, 0.66527644, 0.97282457, 0.40755811,
        0.39594637, 0.4371293 , 0.63436135, 0.28924616, 0.82359445,
        0.07667993, 0.8241387 , 0.15973151, 0.51050315, 0.1186015 ,
        0.3548533 , 0.25114659, 0.44116845, 0.39079616, 0.23344579,
        0.92875055, 0.14759959, 0.25859377, 0.60118208, 0.69154782,
        0.22774673, 0.04335412, 0.85383021, 0.81747365, 0.98008646,
        0.61829183, 0.09613495, 0.88478902, 0.67388038, 0.44970993,
        0.60526701, 0.83140682, 0.21803312, 0.8039053 , 0.89048563,
        0.06490831, 0.54938267, 0.63146686, 0.40217226, 0.65209438,
        0.75770723, 0.44349971, 0.64703594, 0.6041466 , 0.50771241,
        0.97700557, 0.62558788, 0.14158688, 0.17576635, 0.05486669,
        0.9914274 , 0.02803019, 0.54864102, 0.96

In [38]:
wl_name = 'WL_1.5'
sim_names = 'sim1-1'
key = (wl_name, sim_name)
print(len(testing_all_data_og)) # number of columns
print(len(testing_all_data_og[1])) # number of rows
print(len(profile_data[key][:, hour]))

48
365
365


In [39]:
all_data_stacked_og = np.column_stack(testing_all_data_og)
print(len(all_data_stacked_og)) # there are 365 arrays in this baby
print(len(all_data_stacked_og[1])) # each one is a row of data

365
48


In [40]:
all_data_stacked_og

array([[0.40848229, 0.17817066, 0.90451474, ..., 0.9217287 , 0.45816028,
        0.62427049],
       [0.76244866, 0.76912623, 0.71711057, ..., 0.76921168, 0.30384091,
        0.10195329],
       [0.71160393, 0.28762032, 0.11323741, ..., 0.04341932, 0.52674926,
        0.41990886],
       ...,
       [0.30892793, 0.67033454, 0.31205013, ..., 0.01507693, 0.39671696,
        0.03083424],
       [0.22665784, 0.41708502, 0.10141079, ..., 0.65760912, 0.98415018,
        0.5581015 ],
       [0.44245287, 0.29111502, 0.85468285, ..., 0.44936602, 0.26997238,
        0.88543309]])

In [41]:
#!#!#!#! end  _stack_profile_data

df_profile_og =  pd.DataFrame(
    all_data_stacked_og,
    columns=multi_cols,
    index=np.arange(1, days_in_year + 1, 1),
)

In [42]:
df_profile_og

Hour              1                   2                   3             \
Simulation    sim1-1    sim2-2    sim1-1    sim2-2    sim1-1    sim2-2   
1           0.408482  0.178171  0.904515  0.579303  0.253299  0.028625   
2           0.762449  0.769126  0.717111  0.870798  0.986172  0.697765   
3           0.711604  0.287620  0.113237  0.876078  0.930171  0.602602   
4           0.670954  0.303957  0.648364  0.488548  0.973934  0.753285   
5           0.222457  0.936901  0.785936  0.231107  0.140852  0.999778   
..               ...       ...       ...       ...       ...       ...   
361         0.295308  0.578700  0.320887  0.660592  0.654124  0.419703   
362         0.692555  0.358140  0.037695  0.972795  0.659405  0.885947   
363         0.308928  0.670335  0.312050  0.747238  0.633440  0.148685   
364         0.226658  0.417085  0.101411  0.235289  0.381886  0.989430   
365         0.442453  0.291115  0.854683  0.161756  0.400081  0.596745   

Hour              4                   5             ...        20            \
Simulation    sim1-1    sim2-2    sim1-1    sim2-2  ...    sim1-1    sim2-2   
1           0.207945  0.867884  0.499047  0.451148  ...  0.772653  0.400268   
2           0.560468  0.203038  0.286302  0.451566  ...  0.408642  0.518730   
3           0.298799  0.125903  0.316901  0.320058  ...  0.814256  0.013086   
4           0.902251  0.254918  0.596088  0.450139  ...  0.929455  0.941206   
5           0.218507  0.879504  0.806570  0.885165  ...  0.501585  0.076306   
..               ...       ...       ...       ...  ...       ...       ...   
361         0.087153  0.992374  0.666429  0.866987  ...  0.790052  0.548944   
362         0.294101  0.850385  0.408272  0.378239  ...  0.623382  0.219046   
363         0.100707  0.868342  0.872434  0.822646  ...  0.539988  0.260153   
364         0.244442  0.929302  0.478604  0.566048  ...  0.747009  0.012198   
365         0.457267  0.526568  0.747175  0.677636  ...  0.366254  0.141865   

Hour              21                  22                  23            \
Simulation    sim1-1    sim2-2    sim1-1    sim2-2    sim1-1    sim2-2   
1           0.001315  0.591157  0.764004  0.144404  0.020690  0.921729   
2           0.484766  0.176914  0.022725  0.312190  0.838997  0.769212   
3           0.491349  0.237592  0.432162  0.827255  0.972998  0.043419   
4           0.934132  0.204406  0.446037  0.268938  0.362234  0.160731   
5           0.428957  0.807469  0.761016  0.157328  0.366105  0.725328   
..               ...       ...       ...       ...       ...       ...   
361         0.642739  0.144053  0.765392  0.364054  0.845916  0.064109   
362         0.407096  0.183223  0.725326  0.023520  0.543730  0.625516   
363         0.353093  0.419085  0.958452  0.969924  0.924829  0.015077   
364         0.835692  0.494721  0.237328  0.516497  0.917482  0.657609   
365         0.429367  0.465304  0.818906  0.226706  0.465821  0.449366   

Hour              24            
Simulation    sim1-1    sim2-2  
1           0.458160  0.624270  
2           0.303841  0.101953  
3           0.526749  0.419909  
4           0.009857  0.789390  
5           0.592155  0.702195  
..               ...       ...  
361         0.962576  0.730059  
362         0.794383  0.248399  
363         0.396717  0.030834  
364         0.984150  0.558101  
365         0.269972  0.885433  

[365 rows x 48 columns]

Ok, ok, I think I see? We want the final all_data to be an array of 8760 arrays, each one the length of a single simulation. This seems, superfluous.

In [43]:
#profile_data = profile_data_reshaped_multi_sim
profile_data = profile_data_1d_multi_sim

#! explode each element 

#!#!  start _construct_profile_dataframe
sim_label_func=_get_simulation_label
# df_profile = _construct_profile_dataframe(
#     profile_data=profile_data,
#     warming_levels=warming_levels,
#     simulations=simulations,
#     sim_label_func=_get_simulation_label,
#     days_in_year=days_in_year,
#     hours_per_day=hours_per_day,
# )

hours = np.arange(1, 25, 1)
n_warming_levels = len(warming_levels)
n_simulations = len(simulations)


#!#!#! start  _create_single_wl_multi_sim_dataframe
warming_level = warming_levels[0]

# df_profile = _create_single_wl_multi_sim_dataframe(
#     profile_data,
#     warming_levels[0],
#     simulations,
#     sim_label_func,
#     days_in_year,
#     hours,
#     hours_per_day,
# )

wl = warming_level
sim_names = [sim_label_func(sim, i) for i, sim in enumerate(simulations)]

# Ensure simulation names are unique
if len(sim_names) != len(set(sim_names)):
    print(
        "   ⚠️  Warning: Duplicate simulation names detected, adding uniqueness suffixes"
    )
    unique_sim_names = []
    name_counts = {}
    for name in sim_names:
        if name not in name_counts:
            name_counts[name] = 0
            unique_sim_names.append(name)
        else:
            name_counts[name] += 1
            unique_sim_names.append(f"{name}_v{name_counts[name]}")
    sim_names = unique_sim_names

# Create MultiIndex columns
col_tuples = [(hour, sim_name) for hour in hours for sim_name in sim_names]
multi_cols = pd.MultiIndex.from_tuples(col_tuples, names=["Hour", "Simulation"])

# Stack data
#!#!#!#! start  _stack_profile_data
wl_names=[f"WL_{wl}"]
hour_first = True
three_level = False
# all_data = _stack_profile_data(
#     profile_data=profile_data,
#     hours_per_day=hours_per_day,
#     wl_names=[f"WL_{wl}"],
#     sim_names=sim_names,
#     hour_first=True,
# )

all_data = []

In [44]:
hours_in_year

8760

In [45]:
key

('WL_1.5', 'sim2-2')

In [46]:
profile_data[key]

array([0.17817066, 0.57930304, 0.02862475, ..., 0.22670595, 0.44936602,
       0.88543309])

In [47]:
for wl_name in wl_names:
    for sim_name in sim_names:
        key = (wl_name, sim_name)
        all_data.append(profile_data[key])

In [48]:
all_data_stacked = np.column_stack(all_data)

In [49]:
print(len(all_data_stacked))
print(len(all_data_stacked[1]))

8760
2


In [50]:
col_tuples = [(sim_name) for hour in hours for sim_name in sim_names]
multi_cols = pd.MultiIndex.from_tuples(col_tuples, names=["Hour", "Simulation"])

ValueError: Length of names must match number of levels in MultiIndex.

In [ ]:
simulations

['sim1', 'sim2']

In [168]:
df_profile =  pd.DataFrame(
    all_data_stacked,
    columns=simulations,
    index=np.arange(1, hours_in_year + 1, 1),
)

In [169]:
df_profile

,sim1,sim2
1,0.404110,0.275614
2,0.515058,0.567071
3,0.597540,0.639087
4,0.644646,0.862521
5,0.726974,0.018835
...,...,...
8756,0.642255,0.975349
8757,0.976110,0.425952
8758,0.147540,0.808576
8759,0.109547,0.059712


#### Multi sim, multi wl

In [57]:
time_delta = pd.date_range(
    "2020-01-01", periods=8760, freq="h"
)  # 1 year of hourly data
warming_levels = [1.5,2.0]
simulations = ["sim1","sim2"]

# Create test data with proper dimensions
data = np.random.rand(len(warming_levels), len(time_delta), len(simulations))

sample_data_multi_sim_multi_wl = xr.DataArray(
    data,
    dims=["warming_level", "time_delta", "simulation"],
    coords={
        "warming_level": warming_levels,
        "time_delta": time_delta,
        "simulation": simulations,
    },
    attrs={"units": "degC", "variable_id": "tasmax"},
)

profile_data_reshaped_multi_sim_multi_wl, profile_data_1d_multi_sim_multi_wl = compute_profile(sample_data_multi_sim_multi_wl, days_in_year=365, q=0.5)

      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 2 warming level(s) and 2 simulation(s)


      Computing profiles:   0%|          | 0/4 [00:00<?, ?combo/s]

In [ ]:
days_in_year=365
q=0.5
data = sample_data_multi_sim_multi_wl

# Check for simulation dimension
has_simulation = "simulation" in data.dims
if has_simulation:
    n_simulations = len(data.simulation)
    simulations = data.simulation.values
else:
    n_simulations = 1
    simulations = [None]

# Get all available time_delta data (all 30 years)
hours_per_day = 24
hours_per_year = 8760
total_hours = len(data.time_delta)
n_years = total_hours // hours_per_year

print(f"      📊 Processing {total_hours:,} hours ({n_years} years) of data")
print(f"      🎯 Computing {q*100:.0f}th percentile for each hour of year")

# Create hour-of-year coordinate for all data (cycling through 1-8760)
hour_of_year_all = np.tile(np.arange(1, hours_per_year + 1), n_years)[:total_hours]
data = data.assign_coords(hour_of_year=("time_delta", hour_of_year_all))

warming_levels = data.warming_level.values

# Create helper function to extract meaningful simulation labels
def _get_simulation_label(sim: str | int | None, sim_idx: int) -> str:
    """Extract meaningful simulation label from simulation identifier."""
    if sim is None:
        return f"Sim_{sim_idx+1}"

    sim_str = str(sim)
    if "WRF_" in sim_str:
        # Parse simulation name format: WRF_GCM_params_scenario
        # Example: WRF_CESM2_r11i1p1f1_historical+ssp245
        parts = sim_str.split("_")
        if len(parts) >= 4:
            gcm = parts[1]  # e.g., CESM2, CNRM-ESM2-1
            params = parts[2]  # e.g., r11i1p1f1
            scenario = parts[3]  # e.g., historical+ssp245

            # Extract SSP from scenario (e.g., ssp245 from historical+ssp245)
            if "ssp" in scenario:
                ssp_part = scenario.split("ssp")[-1]  # Get part after 'ssp'
                ssp = f"ssp{ssp_part}"
            else:
                ssp = "hist"  # fallback for historical-only

            return f"{gcm}-{params}-{ssp}"
        elif len(parts) >= 2:
            # Fallback for shorter format
            return f"{parts[1]}-{sim_idx+1}"
        else:
            return f"Sim_{sim_idx+1}"
    else:
        # Ensure uniqueness by adding index for non-WRF format
        base_name = sim_str.split("_")[0] if "_" in sim_str else sim_str
        return f"{base_name}-{sim_idx+1}"

# Process all data using quantile computation across years
print(
    f"      ⚙️ Computing quantiles for {len(warming_levels)} warming level(s) and {n_simulations} simulation(s)"
)

# Eagerly compute all dask data at once (one round-trip to scheduler)
if hasattr(data.data, "chunks"):
    print("      📥 Loading data into memory...")
    from dask.diagnostics import ProgressBar

    with ProgressBar():
        data = data.compute()

      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 2 warming level(s) and 2 simulation(s)


In [61]:
profile_data = profile_data_1d_multi_sim_multi_wl
#profile_data = profile_data_reshaped_multi_sim_multi_wl

In [ ]:
#!#!#! start  _create_multi_wl_multi_sim_dataframe

# df_profile = _create_multi_wl_multi_sim_dataframe(
#     profile_data,
#     warming_levels,
#     simulations,
#     sim_label_func,
#     days_in_year,
#     hours,
#     hours_per_day,
# )

wl_names = [f"WL_{wl}" for wl in warming_levels]
sim_names = [sim_label_func(sim, i) for i, sim in enumerate(simulations)]

# Ensure simulation names are unique
if len(sim_names) != len(set(sim_names)):
    print(
        "   ⚠️  Warning: Duplicate simulation names detected, adding uniqueness suffixes"
    )
    unique_sim_names = []
    name_counts = {}
    for name in sim_names:
        if name not in name_counts:
            name_counts[name] = 0
            unique_sim_names.append(name)
        else:
            name_counts[name] += 1
            unique_sim_names.append(f"{name}_v{name_counts[name]}")
    sim_names = unique_sim_names

# Create MultiIndex columns
col_tuples = [
    (wl_name, sim_name)
    for wl_name in wl_names
    for sim_name in sim_names
]
multi_cols = pd.MultiIndex.from_tuples(
    col_tuples, names=["Warming_Level", "Simulation"]
)


#!#!#!#! start _stack_profile_data
# Stack data
hour_first=True
three_level=True
# all_data = _stack_profile_data(
#     profile_data=profile_data,
#     hours_per_day=hours_per_day,
#     wl_names=wl_names,
#     sim_names=sim_names,
#     hour_first=True,
#     three_level=True,
# )
all_data = []

# For three-level index: iterate hour -> wl -> sim
for wl_name in wl_names:
    for sim_name in sim_names:
        key = (wl_name, sim_name)
        all_data.append(profile_data[key])
        print(all_data)

all_data = np.column_stack(all_data)

#!#!#!#! end _stack_profile_data

df_profile = pd.DataFrame(
    all_data,
    columns=multi_cols,
    index=np.arange(1, hours_in_year + 1, 1),
)

#!#!#! end  _create_single_wl_multi_sim_dataframe

#!#! end _construct_profile_dataframe



### Exploded final function

Alright, I see it now. Let's go from the top.

In [22]:
time_delta = pd.date_range(
    "2020-01-01", periods=8760, freq="h"
)  # 1 year of hourly data
warming_levels = [1.5]
warming_levels_2 = [1.5, 2.0]
simulations = ["sim1"]
simulations_2 = ["sim1","sim2"]

# Create test data with proper dimensions
data = np.random.rand(len(warming_levels), len(time_delta), len(simulations))

simple_data = xr.DataArray(
    data,
    dims=["warming_level", "time_delta", "simulation"],
    coords={
        "warming_level": warming_levels,
        "time_delta": time_delta,
        "simulation": simulations,
    },
    attrs={"units": "degC", "variable_id": "tasmax"},
)

# Create test data with proper dimensions
data = np.random.rand(len(warming_levels_2), len(time_delta), len(simulations))
multi_wl_data = xr.DataArray(
    data,
    dims=["warming_level", "time_delta", "simulation"],
    coords={
        "warming_level": warming_levels_2,
        "time_delta": time_delta,
        "simulation": simulations,
    },
    attrs={"units": "degC", "variable_id": "tasmax"},
)

# Create test data with proper dimensions
data = np.random.rand(len(warming_levels), len(time_delta), len(simulations_2))
multi_sim_data = xr.DataArray(
    data,
    dims=["warming_level", "time_delta", "simulation"],
    coords={
        "warming_level": warming_levels,
        "time_delta": time_delta,
        "simulation": simulations_2,
    },
    attrs={"units": "degC", "variable_id": "tasmax"},
)

# Create test data with proper dimensions
data = np.random.rand(len(warming_levels_2), len(time_delta), len(simulations_2))
multi_both_data = xr.DataArray(
    data,
    dims=["warming_level", "time_delta", "simulation"],
    coords={
        "warming_level": warming_levels_2,
        "time_delta": time_delta,
        "simulation": simulations_2,
    },
    attrs={"units": "degC", "variable_id": "tasmax"},
)


In [ ]:
def _stack_profile_data(
    profile_data: dict,
    hours_per_day: int,
    wl_names: list,
    sim_names: list,
    hour_first: bool = True,
    three_level: bool = False,
) -> np.ndarray:
    """
    Stack profile data into a single array for DataFrame construction.

    Parameters
    ----------
    profile_data : dict
        Dictionary with (wl_name, sim_name) keys and profile arrays as values
    hours_per_day : int
        Number of hours per day (24)
    wl_names : list
        List of warming level names
    sim_names : list
        List of simulation names
    hour_first : bool, optional
        Whether hour should be the first level in iteration order
    three_level : bool, optional
        Whether this is a three-level MultiIndex (Hour, WL, Sim)

    Returns
    -------
    np.ndarray
        Stacked data array ready for DataFrame construction
    """
    all_data = []

    if three_level:
        # For three-level index: iterate hour -> wl -> sim
        for wl_name in wl_names:
            for sim_name in sim_names:
                key = (wl_name, sim_name)
                all_data.append(profile_data[key])
    elif hour_first:
        # For two-level with hour first
        for wl_name in wl_names:
            for sim_name in sim_names:
                key = (wl_name, sim_name)
                all_data.append(profile_data[key])
    else:
        # For other two-level cases
        for wl_name in wl_names:
            for sim_name in sim_names:
                    key = (wl_name, sim_name)
                    all_data.append(profile_data[key])

    return np.column_stack(all_data)

In [24]:
def _create_simple_dataframe(
    profile_data: dict,
    warming_level: float,
    simulation: any,
    sim_label_func: callable,
    days_in_year: int,
    hours: np.ndarray,
    hours_per_day: int,
    hours_per_year: int,
) -> pd.DataFrame:
    """
    Create a simple DataFrame for single warming level and single simulation.

    Parameters
    ----------
    profile_data : dict
        Profile data dictionary
    warming_level : float
        Single warming level value
    simulation : any
        Single simulation identifier
    sim_label_func : callable
        Function to get simulation label
    days_in_year : int
        Number of days in year
    hours : np.ndarray
        Array of hour values
    hours_per_day : int
        Hours per day (24)

    Returns
    -------
    pd.DataFrame
        Simple DataFrame with hour columns
    """

    wl_key = warming_level
    sim_key = sim_label_func(simulation, 0)

    # Stack data
    all_data = _stack_profile_data(
            profile_data=profile_data,
            hours_per_day=hours_per_day,
            wl_names=[f"WL_{wl_key}"],
            sim_names=[sim_key],
            hour_first=True,
        )

    return  pd.DataFrame(
            all_data,
            columns=[sim_key],
            index=np.arange(1, hours_per_year + 1, 1),
        )



In [ ]:
def _create_single_wl_multi_sim_dataframe(
    profile_data: dict,
    warming_level: float,
    simulations: list,
    sim_label_func: callable,
    days_in_year: int,
    hours: np.ndarray,
    hours_per_day: int,
    hours_per_year: int,
) -> pd.DataFrame:
    """
    Create DataFrame for single warming level with multiple simulations.

    Parameters
    ----------
    profile_data : dict
        Profile data dictionary
    warming_level : float
        Single warming level value
    simulations : list
        List of simulation identifiers
    sim_label_func : callable
        Function to get simulation labels
    days_in_year : int
        Number of days in year
    hours : np.ndarray
        Array of hour values
    hours_per_day : int
        Hours per day (24)

    Returns
    -------
    pd.DataFrame
        DataFrame with (Hour, Simulation) MultiIndex columns
    """
    wl = warming_level
    sim_names = [sim_label_func(sim, i) for i, sim in enumerate(simulations)]

    # Ensure simulation names are unique
    if len(sim_names) != len(set(sim_names)):
        print(
            "   ⚠️  Warning: Duplicate simulation names detected, adding uniqueness suffixes"
        )
        unique_sim_names = []
        name_counts = {}
        for name in sim_names:
            if name not in name_counts:
                name_counts[name] = 0
                unique_sim_names.append(name)
            else:
                name_counts[name] += 1
                unique_sim_names.append(f"{name}_v{name_counts[name]}")
        sim_names = unique_sim_names

    # Stack data
    all_data = _stack_profile_data(
        profile_data=profile_data,
        hours_per_day=hours_per_day,
        wl_names=[f"WL_{wl}"],
        sim_names=sim_names,
        hour_first=True,
    )
    
    return pd.DataFrame(
            all_data,
            columns=simulations,
            index=np.arange(1, hours_per_year + 1, 1),
        )




In [34]:
def _create_multi_wl_single_sim_dataframe(
    profile_data: dict,
    warming_levels: np.ndarray,
    simulation: any,
    sim_label_func: callable,
    days_in_year: int,
    hours: np.ndarray,
    hours_per_day: int,
    hours_per_year: int,
) -> pd.DataFrame:
    """
    Create DataFrame for multiple warming levels with single simulation.

    Parameters
    ----------
    profile_data : dict
        Profile data dictionary
    warming_levels : np.ndarray
        Array of warming level values
    simulation : any
        Single simulation identifier
    sim_label_func : callable
        Function to get simulation label
    days_in_year : int
        Number of days in year
    hours : np.ndarray
        Array of hour values
    hours_per_day : int
        Hours per day (24)

    Returns
    -------
    pd.DataFrame
        DataFrame with (Hour, Warming_Level) MultiIndex columns
    """
    sim_name = sim_label_func(simulation, 0)
    wl_names = [f"WL_{wl}" for wl in warming_levels]

    # Create MultiIndex columns
    col_tuples = [(hour, wl_name) for hour in hours for wl_name in wl_names]
    multi_cols = pd.MultiIndex.from_tuples(col_tuples, names=["Hour", "Warming_Level"])

    # Stack data
    all_data = _stack_profile_data(
        profile_data=profile_data,
        hours_per_day=hours_per_day,
        wl_names=wl_names,
        sim_names=[sim_name],
        hour_first=True,
    )

    return pd.DataFrame(
        all_data,
        columns=multi_cols,
        index=np.arange(1, hours_per_year + 1, 1),
    )


In [35]:
def _create_multi_wl_multi_sim_dataframe(
    profile_data: dict,
    warming_levels: np.ndarray,
    simulations: list,
    sim_label_func: callable,
    days_in_year: int,
    hours: np.ndarray,
    hours_per_day: int,
    hours_per_year: int,
) -> pd.DataFrame:
    """
    Create DataFrame for multiple warming levels and multiple simulations.

    Parameters
    ----------
    profile_data : dict
        Profile data dictionary
    warming_levels : np.ndarray
        Array of warming level values
    simulations : list
        List of simulation identifiers
    sim_label_func : callable
        Function to get simulation labels
    days_in_year : int
        Number of days in year
    hours : np.ndarray
        Array of hour values
    hours_per_day : int
        Hours per day (24)

    Returns
    -------
    pd.DataFrame
        DataFrame with (Hour, Warming_Level, Simulation) MultiIndex columns
    """
    wl_names = [f"WL_{wl}" for wl in warming_levels]
    sim_names = [sim_label_func(sim, i) for i, sim in enumerate(simulations)]

    # Ensure simulation names are unique
    if len(sim_names) != len(set(sim_names)):
        print(
            "   ⚠️  Warning: Duplicate simulation names detected, adding uniqueness suffixes"
        )
        unique_sim_names = []
        name_counts = {}
        for name in sim_names:
            if name not in name_counts:
                name_counts[name] = 0
                unique_sim_names.append(name)
            else:
                name_counts[name] += 1
                unique_sim_names.append(f"{name}_v{name_counts[name]}")
        sim_names = unique_sim_names

    # Create MultiIndex columns
    col_tuples = [
        (wl_name, sim_name)
        for wl_name in wl_names
        for sim_name in sim_names
    ]
    multi_cols = pd.MultiIndex.from_tuples(
        col_tuples, names=["Warming_Level", "Simulation"]
    )

    #!#!#!#! start _stack_profile_data
    # Stack data
    all_data = _stack_profile_data(
        profile_data=profile_data,
        hours_per_day=hours_per_day,
        wl_names=wl_names,
        sim_names=sim_names,
        hour_first=True,
        three_level=True,
    )


    return pd.DataFrame(
        all_data,
        columns=multi_cols,
        index=np.arange(1, hours_per_year + 1, 1),
    )

In [36]:
def _construct_profile_dataframe(
    profile_data: dict,
    warming_levels: np.ndarray,
    simulations: list,
    sim_label_func: callable,
    days_in_year: int,
    hours_per_day: int,
    hours_per_year: int,
) -> pd.DataFrame:
    """
    Construct a DataFrame from profile data based on warming level and simulation dimensions.

    Parameters
    ----------
    profile_data : dict
        Dictionary with (warming_level, simulation) keys and profile arrays as values
    warming_levels : np.ndarray
        Array of warming level values
    simulations : list
        List of simulation identifiers
    sim_label_func : callable
        Function to extract simulation labels
    days_in_year : int
        Number of days in the year (365 or 366)
    hours_per_day : int
        Number of hours per day (24)

    Returns
    -------
    pd.DataFrame
        Structured DataFrame with appropriate column structure based on dimensions
    """
    hours = np.arange(1, 25, 1)
    n_warming_levels = len(warming_levels)
    n_simulations = len(simulations)

    if n_warming_levels == 1 and n_simulations == 1:
        return _create_simple_dataframe(
            profile_data,
            warming_levels[0],
            simulations[0],
            sim_label_func,
            days_in_year,
            hours,
            hours_per_day,
            hours_per_year
        )
    elif n_warming_levels == 1 and n_simulations > 1:
    
        return _create_single_wl_multi_sim_dataframe(
            profile_data,
            warming_levels[0],
            simulations,
            sim_label_func,
            days_in_year,
            hours,
            hours_per_day,
            hours_per_year
        )
        
    elif n_warming_levels > 1 and n_simulations == 1:
        return _create_multi_wl_single_sim_dataframe(
            profile_data,
            warming_levels,
            simulations[0],
            sim_label_func,
            days_in_year,
            hours,
            hours_per_day,
            hours_per_year
        )
    else:
        return _create_multi_wl_multi_sim_dataframe(
            profile_data,
            warming_levels,
            simulations,
            sim_label_func,
            days_in_year,
            hours,
            hours_per_day,
            hours_per_year
        )


In [37]:
def test_compute_profile(data: xr.DataArray, days_in_year: int = 365, q=0.5) -> pd.DataFrame:
    """
    Calculates the standard year climate profile for warming level data using 8760
    analysis.

    This function handles global warming levels approach using time_delta coordinate.
    Processes all 30 years of warming level data centered around the year a warming level
    is reached, computes the specified quantile for each hour of the year across all years,
    then selects the actual data value closest to that quantile (not interpolated),
    and returns a characteristic profile of 8760 hours (one year) for each warming level
    and simulation combination.

    Parameters
    ----------
    data : xr.DataArray
        Hourly base-line subtracted data for one variable with warming_level,
        time_delta, and simulation dimensions. Expected to contain ~30 years
        (262,800 hours) of data for each warming level and simulation.

    days_in_year : int, optional
        Either 366 or 365, depending on whether or not the year is a leap year.
        Default to 365 days

    q : float, optional
        Quantile value for selecting representative values (0.0 to 1.0).
        Default is 0.5 (median).

    Returns
    -------
    pd.DataFrame
        Standard year table for each warming level and simulation,
        with days of year as the index and hour of day as the columns.
        Multi-index columns include Hour, Warming_Level, and Simulation dimensions.

    """
    # Check for simulation dimension
    has_simulation = "simulation" in data.dims
    if has_simulation:
        n_simulations = len(data.simulation)
        simulations = data.simulation.values
    else:
        n_simulations = 1
        simulations = [None]

    # Get all available time_delta data (all 30 years)
    hours_per_day = 24
    hours_per_year = 8760
    total_hours = len(data.time_delta)
    n_years = total_hours // hours_per_year

    print(f"      📊 Processing {total_hours:,} hours ({n_years} years) of data")
    print(f"      🎯 Computing {q*100:.0f}th percentile for each hour of year")

    # Create hour-of-year coordinate for all data (cycling through 1-8760)
    hour_of_year_all = np.tile(np.arange(1, hours_per_year + 1), n_years)[:total_hours]
    data = data.assign_coords(hour_of_year=("time_delta", hour_of_year_all))

    warming_levels = data.warming_level.values

    # Create helper function to extract meaningful simulation labels
    def _get_simulation_label(sim: str | int | None, sim_idx: int) -> str:
        """Extract meaningful simulation label from simulation identifier."""
        if sim is None:
            return f"Sim_{sim_idx+1}"

        sim_str = str(sim)
        if "WRF_" in sim_str:
            # Parse simulation name format: WRF_GCM_params_scenario
            # Example: WRF_CESM2_r11i1p1f1_historical+ssp245
            parts = sim_str.split("_")
            if len(parts) >= 4:
                gcm = parts[1]  # e.g., CESM2, CNRM-ESM2-1
                params = parts[2]  # e.g., r11i1p1f1
                scenario = parts[3]  # e.g., historical+ssp245

                # Extract SSP from scenario (e.g., ssp245 from historical+ssp245)
                if "ssp" in scenario:
                    ssp_part = scenario.split("ssp")[-1]  # Get part after 'ssp'
                    ssp = f"ssp{ssp_part}"
                else:
                    ssp = "hist"  # fallback for historical-only

                return f"{gcm}-{params}-{ssp}"
            elif len(parts) >= 2:
                # Fallback for shorter format
                return f"{parts[1]}-{sim_idx+1}"
            else:
                return f"Sim_{sim_idx+1}"
        else:
            # Ensure uniqueness by adding index for non-WRF format
            base_name = sim_str.split("_")[0] if "_" in sim_str else sim_str
            return f"{base_name}-{sim_idx+1}"

    # Process all data using quantile computation across years
    print(
        f"      ⚙️ Computing quantiles for {len(warming_levels)} warming level(s) and {n_simulations} simulation(s)"
    )

    # Eagerly compute all dask data at once (one round-trip to scheduler)
    if hasattr(data.data, "chunks"):
        print("      📥 Loading data into memory...")
        from dask.diagnostics import ProgressBar

        with ProgressBar():
            data = data.compute()

    # Initialize storage for profiles
    profile_data = {}

    # Progress tracking
    total_combinations = len(warming_levels) * n_simulations
    with tqdm(
        total=total_combinations,
        desc="      Computing profiles",
        unit="combo",
        leave=False,
    ) as pbar:

        for wl_idx, wl in enumerate(warming_levels):
            for sim_idx, sim in enumerate(simulations):
                # Get simulation label
                sim_label = _get_simulation_label(sim, sim_idx)

                # Select data for this warming level and simulation combination
                if has_simulation:
                    subset_data = data.isel(warming_level=wl_idx, simulation=sim_idx)
                else:
                    subset_data = data.isel(warming_level=wl_idx)

                # Vectorized quantile computation using numpy
                # Reshape raw values into (n_years, hours_per_year) then compute
                # the quantile across years for each hour-of-year position
                values = subset_data.values
                n_total = len(values)
                usable = (n_total // hours_per_year) * hours_per_year
                year_hour_matrix = values[:usable].reshape(-1, hours_per_year)

                # Compute quantile targets for each of the 8760 hour positions
                quantile_targets = np.nanquantile(
                    year_hour_matrix, q, axis=0
                )  # shape: (8760,)

                # For each hour position, find the actual year whose value is
                # closest to the quantile (avoids interpolation)
                diffs = np.abs(
                    year_hour_matrix - quantile_targets[np.newaxis, :]
                )  # (n_years, 8760)
                closest_year_idx = np.nanargmin(diffs, axis=0)  # (8760,)
                profile_1d = year_hour_matrix[
                    closest_year_idx, np.arange(hours_per_year)
                ]

                # Store the profile
                key = (f"WL_{wl}", sim_label)
                profile_data[key] = profile_1d

                pbar.update(1)

    df_profile = _construct_profile_dataframe(
        profile_data=profile_data,
        warming_levels=warming_levels,
        simulations=simulations,
        sim_label_func=_get_simulation_label,
        days_in_year=days_in_year,
        hours_per_day=hours_per_day,
        hours_per_year=hours_per_year
    )

    # #! -start
    # df_profile_reshaped = _construct_profile_dataframe(
    #     profile_data=profile_data_reshaped,
    #     warming_levels=warming_levels,
    #     simulations=simulations,
    #     sim_label_func=_get_simulation_label,
    #     days_in_year=days_in_year,
    #     hours_per_day=hours_per_day,
    # )
    # df_profile_1d = _construct_profile_dataframe(
    #     profile_data=profile_data_1d,
    #     warming_levels=warming_levels,
    #     simulations=simulations,
    #     sim_label_func=_get_simulation_label,
    #     days_in_year=days_in_year,
    #     hours_per_day=hours_per_day,
    # )
    # #! -end

    # # Determine which formatting function to use based on the structure
    # _format_based_on_structure(df_profile)

    # #! -start
    # _format_based_on_structure(df_profile_reshaped)
    # _format_based_on_structure(df_profile_1d)
    # #! -end

    # # Prepare metadata dictionary
    # metadata = {
    #     "quantile": q,
    #     "method": "8760 analysis - actual data closest to quantile across 30 years",
    #     "description": f"Climate profile computed using actual data values closest to the {q*100:.0f}th percentile of hourly data",
    # }

    # # Add original data attributes if available
    # if hasattr(data, "attrs"):
    #     if "units" in data.attrs:
    #         metadata["units"] = data.attrs["units"]
    #     if "extended_description" in data.attrs:
    #         metadata["extended_description"] = data.attrs["extended_description"]
    #     if "variable_id" in data.attrs:
    #         metadata["variable_name"] = data.attrs["variable_id"]
    #     elif hasattr(data, "name") and data.name:
    #         metadata["variable_name"] = data.name

    # # Set all metadata using the helper function
    # set_profile_metadata(df_profile, metadata)

    # #! -start
    # set_profile_metadata(df_profile_reshaped, metadata)
    # set_profile_metadata(df_profile_1d, metadata)
    # #! -end

    # print(f"      ✅ Profile computation complete! Final shape: {df_profile.shape}")
    # print(
    #     f"         With index: {df_profile.index.name}, columns: {df_profile.columns.names}"
    # )
    # if hasattr(data, "attrs") and "units" in data.attrs:
    #     print(f"         Units: {data.attrs['units']}")

    return df_profile #profile_data_reshaped, profile_data_1d

In [ ]:
simple_data
multi_both_data
multi_sim_data
multi_wl_data

In [38]:
test_simple = test_compute_profile(simple_data,days_in_year=365, q=0.5)
test_multi_sim = test_compute_profile(multi_sim_data,days_in_year=365, q=0.5)
test_multi_both = test_compute_profile(multi_both_data,days_in_year=365, q=0.5)

      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 1 warming level(s) and 1 simulation(s)


      Computing profiles:   0%|          | 0/1 [00:00<?, ?combo/s]

      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 1 warming level(s) and 2 simulation(s)


      Computing profiles:   0%|          | 0/2 [00:00<?, ?combo/s]

      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 2 warming level(s) and 2 simulation(s)


      Computing profiles:   0%|          | 0/4 [00:00<?, ?combo/s]

In [39]:
test_multi_wl = test_compute_profile(multi_wl_data,days_in_year=365, q=0.5)

      📊 Processing 8,760 hours (1 years) of data
      🎯 Computing 50th percentile for each hour of year
      ⚙️ Computing quantiles for 2 warming level(s) and 1 simulation(s)


      Computing profiles:   0%|          | 0/2 [00:00<?, ?combo/s]

ValueError: Shape of passed values is (8760, 2), indices imply (8760, 48)

In [33]:
test_multi_both

Warming_Level    WL_1.5              WL_2.0          
Simulation       sim1-1    sim2-2    sim1-1    sim2-2
1              0.756836  0.815919  0.607653  0.148704
2              0.998678  0.691584  0.327121  0.070381
3              0.472953  0.610093  0.912285  0.164423
4              0.441115  0.210421  0.282178  0.191729
5              0.899630  0.944791  0.245240  0.536671
...                 ...       ...       ...       ...
8756           0.635748  0.546102  0.113044  0.407301
8757           0.235903  0.525724  0.008331  0.019155
8758           0.745049  0.641434  0.883221  0.746005
8759           0.552966  0.719345  0.607501  0.873689
8760           0.590266  0.521485  0.999306  0.339217

[8760 rows x 4 columns]

## Testing

In [ ]:
# Set up the Standard Year generator
profile_selections = {  
    ## Required variable and profile arguments
    "variable": "Air Temperature at 2m",
    "resolution": "3 km",
    "q": 0.5,
    "units": "degF",

    ## Required approach arguments, Options: "Warming Level", "Time"
    "approach": "Warming Level",
    # "warming_level": [warming_level], # GWL-option only
    # "centered_year": centered_year, # Time-based option only
    
    ## Required location arguments -- Uncomment your desired location option and modify
    "stations": ["Sacramento Executive Airport (KSAC)"], 
    # "latitude": (latitude - 0.02, latitude + 0.02), 
    # "longitude": (longitude - 0.02, longitude + 0.02),  
    # "cached_area": area_name, 

    ## Additional optional arguments -- Uncomment any desired options and modify
    # "no_delta": True, 
    # "warming_level_window": 10,
    # "time_profile_scenario": "SSP2-4.5,
    # "bias_adjusted_models": True,
}

# Generate the climate profile
profile = get_climate_profile(**profile_selections)

In [ ]:
# the function uses the previously defined profile selections to generate the output file name
export_profile_to_csv(profile, **profile_selections)